# 🩺 MedRAG — AI Medical Assistant (RAG-based LLM System)

> **An end-to-end Retrieval-Augmented Generation (RAG) system built on the Merck Medical Manual — grounding LLM responses in 4,000+ pages of verified clinical knowledge.**

---

## 📋 Notebook Contents

| Section | Description |
|---|---|
| 1. Setup | Install dependencies, import libraries |
| 2. Baseline LLM | Raw LLM responses with no context |
| 3. Prompt Engineering | Structured prompts, no retrieval |
| 4. Data Preparation | Load PDF, chunk, embed, index |
| 5. RAG Pipeline | Context-grounded responses |
| 6. Evaluation | LLM-as-a-Judge (Groundedness + Relevance) |
| 7. Insights | Key findings and business impact |

---

### ⚙️ Runtime Requirements
- Google Colab with **GPU (T4 or better)** recommended
- ~8GB RAM minimum
- ~5GB disk for model + index

---
## 1. 📦 Setup — Install Dependencies

In [ ]:
# ── Step 1a: Install llama-cpp-python with GPU (CUDA) support ──────────────
# For CPU-only: change DLLAMA_CUBLAS=on → DLLAMA_CUBLAS=off
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# NOTE: Restart the runtime after this cell completes, then run from the next cell.

In [ ]:
# ── Step 1b: Install all other dependencies ─────────────────────────────────
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 \
             pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 \
             chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

print("✅ All dependencies installed successfully.")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json, os
import tiktoken
import pandas as pd

# Document loading & chunking
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader

# Embeddings & vector store
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# LLM
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

print("✅ All libraries imported successfully.")

---
## 2. 🤖 Load the LLM — Mistral-7B (GGUF)

We use **Mistral-7B-Instruct** quantized to Q4_K_M (4-bit) via `llama-cpp-python`.
This enables local inference on a T4 GPU with no API costs.

| Parameter | Value | Rationale |
|---|---|---|
| `n_gpu_layers` | 35 | Offload most layers to GPU |
| `n_ctx` | 4096 | Context window size |
| `temperature` | 0.0 | Deterministic — critical for medical use |
| `top_p` | 0.95 | Nucleus sampling |
| `max_tokens` | 512 | Generous output for clinical detail |

In [ ]:
# ── Download and load Mistral-7B-Instruct (GGUF Q4_K_M quantization) ────────
MODEL_REPO_ID = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF"
MODEL_FILENAME = "mistral-7b-instruct-v0.1.Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename=MODEL_FILENAME)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=35,    # Set to 0 for CPU-only
    n_ctx=4096,
    n_threads=8,
    verbose=False
)

print(f"✅ LLM loaded: {MODEL_FILENAME}")

In [ ]:
# ── Helper: Baseline response (no system prompt, no context) ─────────────────
def baseline_response(query, max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    """Generate a raw LLM response with no additional context."""
    output = llm(
        prompt=query,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return output['choices'][0]['text'].strip()

print("✅ Baseline response function defined.")

---
## 3. 🧪 Approach 1 — Baseline LLM (No Context)

We first test the LLM in its raw state — no prompt engineering, no retrieved context.
This establishes our **baseline** to measure improvements from RAG.

> ⚠️ **Expected limitation**: The model may hallucinate clinical details or give generic responses.

In [ ]:
# ── Query 1: Sepsis Protocol ─────────────────────────────────────────────────
q1 = "What is the protocol for managing sepsis in a critical care unit?"
print(f"Query: {q1}\n{'─'*60}")
print(baseline_response(q1))

In [ ]:
# ── Query 2: Appendicitis ─────────────────────────────────────────────────────
q2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed?"
print(f"Query: {q2}\n{'─'*60}")
print(baseline_response(q2))

In [ ]:
# ── Query 3: Alopecia Areata ──────────────────────────────────────────────────
q3 = "What are the effective treatments for sudden patchy hair loss (localized bald spots on the scalp), and what are the possible causes?"
print(f"Query: {q3}\n{'─'*60}")
print(baseline_response(q3))

In [ ]:
# ── Query 4: Traumatic Brain Injury ──────────────────────────────────────────
q4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(f"Query: {q4}\n{'─'*60}")
print(baseline_response(q4))

In [ ]:
# ── Query 5: Leg Fracture ─────────────────────────────────────────────────────
q5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip?"
print(f"Query: {q5}\n{'─'*60}")
print(baseline_response(q5))

---
## 4. 🎯 Approach 2 — Prompt-Engineered LLM

We now add a structured **system prompt** that instructs the model to act as a medical assistant and format its response clearly.

**Goal**: Measure how much prompt engineering alone improves response quality — without any retrieval.

In [ ]:
# ── System prompt for prompt engineering ─────────────────────────────────────
PE_SYSTEM_PROMPT = """You are a concise, expert medical assistant.
Answer the following clinical question clearly and accurately.
Structure your response with:
1. Brief overview
2. Key clinical points
3. Recommended actions or treatments
"""

def pe_response(query, max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    """Generate a response with prompt engineering, no retrieval context."""
    prompt = f"{PE_SYSTEM_PROMPT}\n\nQuestion: {query}\n\nAnswer:"
    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return output['choices'][0]['text'].strip()

print("✅ Prompt-engineered response function defined.")

In [ ]:
print(f"Query: {q1}\n{'─'*60}")
print(pe_response(q1))

In [ ]:
print(f"Query: {q2}\n{'─'*60}")
print(pe_response(q2))

In [ ]:
print(f"Query: {q3}\n{'─'*60}")
print(pe_response(q3))

In [ ]:
print(f"Query: {q4}\n{'─'*60}")
print(pe_response(q4))

In [ ]:
print(f"Query: {q5}\n{'─'*60}")
print(pe_response(q5))

---
## 5. 📚 Data Preparation — Building the RAG Knowledge Base

This section transforms the raw Merck Manual PDF into a searchable vector index.

### Pipeline
```
PDF → Pages → Chunks → Embeddings → ChromaDB Index
```

### Key Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| **Splitter** | RecursiveCharacterTextSplitter | Respects paragraph/sentence boundaries |
| **Chunk size** | 1000 chars | Balances context completeness vs. precision |
| **Overlap** | 200 chars | Prevents context loss at chunk boundaries |
| **Embedding model** | all-MiniLM-L6-v2 | Fast, high-quality, 384-dim |
| **Vector store** | ChromaDB | Persistent, easy local setup |

In [ ]:
# ── 5.1 Load the Merck Manual PDF ────────────────────────────────────────────
PDF_PATH = "/content/merck_manual.pdf"  # Update path as needed

loader = PyMuPDFLoader(PDF_PATH)
documents = loader.load()

print(f"✅ Loaded {len(documents)} pages from the Merck Manual.")
print(f"\n📄 Sample — Page 1 preview:")
print(documents[0].page_content[:500])

In [ ]:
# ── 5.2 Inspect the first 5 pages ────────────────────────────────────────────
print("First 5 pages overview:")
print("─" * 60)
for i, doc in enumerate(documents[:5]):
    print(f"Page {i+1} | Length: {len(doc.page_content)} chars | Source: {doc.metadata.get('source', 'N/A')}")
    print(f"  Preview: {doc.page_content[:150].strip()}...")
    print()

In [ ]:
# ── 5.3 Data statistics ───────────────────────────────────────────────────────
page_lengths = [len(doc.page_content) for doc in documents]

stats = pd.DataFrame({
    'Metric': ['Total Pages', 'Total Characters', 'Avg Chars/Page', 'Min Chars/Page', 'Max Chars/Page'],
    'Value': [
        len(documents),
        sum(page_lengths),
        round(sum(page_lengths) / len(page_lengths), 0),
        min(page_lengths),
        max(page_lengths)
    ]
})
print(stats.to_string(index=False))

In [ ]:
# ── 5.4 Chunk the documents ───────────────────────────────────────────────────
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(documents)
chunk_lengths = [len(c.page_content) for c in chunks]

print(f"✅ Created {len(chunks)} chunks")
print(f"   chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"   Avg chunk length: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")
print(f"\n📝 Sample chunk:")
print(chunks[100].page_content)

In [ ]:
# ── 5.5 Initialize embedding model ───────────────────────────────────────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_fn = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)

# Quick sanity check
test_vec = embedding_fn.embed_query("What is sepsis?")
print(f"✅ Embedding model loaded: '{EMBEDDING_MODEL}'")
print(f"   Vector dimension: {len(test_vec)}")

In [ ]:
# ── 5.6 Build ChromaDB vector store ──────────────────────────────────────────
# ⏱️ This takes several minutes for a 4,000+ page document. Run once.
CHROMA_DIR = "./chroma_db"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_fn,
    persist_directory=CHROMA_DIR,
    collection_name="merck_manual"
)
vector_store.persist()

print(f"✅ Vector store built and persisted to: {CHROMA_DIR}")
print(f"   Total documents indexed: {len(chunks)}")

In [ ]:
# ── 5.7 Create retriever ──────────────────────────────────────────────────────
TOP_K = 3  # Retrieve top-3 most relevant chunks

retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

# Verify retrieval with a test query
test_docs = retriever.get_relevant_documents("sepsis management protocol")
print(f"✅ Retriever initialized (top_k={TOP_K})")
print(f"\n🔍 Test retrieval for 'sepsis management protocol':")
print(f"   Retrieved {len(test_docs)} chunks")
print(f"\n📄 Top result preview:")
print(test_docs[0].page_content[:400])

---
## 6. 🧠 Approach 3 — RAG Pipeline (Grounded Responses)

Now we combine everything:
1. **Retrieve** relevant chunks from ChromaDB
2. **Inject** retrieved context into a structured system prompt
3. **Generate** a grounded, clinically accurate response

The key constraint: the model is explicitly instructed to answer **only from the provided context**.

In [ ]:
# ── RAG system and user prompt templates ─────────────────────────────────────
RAG_SYSTEM_PROMPT = """You are a knowledgeable medical assistant trained on the Merck Manual.
Your role is to provide accurate, evidence-based medical information to healthcare professionals.

CRITICAL RULES:
- Answer ONLY based on the provided medical context.
- If the context does not contain sufficient information, clearly state that.
- Use precise medical terminology.
- Structure your response clearly with relevant clinical details.
- Do NOT fabricate medical facts, drug names, or dosages.
- Always recommend consulting a qualified physician for patient care decisions.
"""

RAG_USER_TEMPLATE = """Use the following medical context to answer the question.

Context:
{context}

Question: {question}

Answer:"""

print("✅ RAG prompt templates defined.")

In [ ]:
# ── RAG response function ─────────────────────────────────────────────────────
def rag_response(query, k=3, max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    """
    Full RAG pipeline:
      1. Retrieve top-k relevant chunks from ChromaDB
      2. Combine chunks into context string
      3. Format prompt with system message + context + question
      4. Generate grounded response via LLM
    """
    # Step 1: Retrieve
    docs = retriever.get_relevant_documents(query=query, k=k)
    context = ". ".join([d.page_content for d in docs])
    
    # Step 2: Format prompt
    user_message = RAG_USER_TEMPLATE.replace('{context}', context).replace('{question}', query)
    full_prompt = RAG_SYSTEM_PROMPT + '\n' + user_message
    
    # Step 3: Generate
    try:
        output = llm(
            prompt=full_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )
        return output['choices'][0]['text'].strip(), context, docs
    except Exception as e:
        return f"Error: {e}", context, docs

print("✅ RAG response function defined.")

In [ ]:
# ── Query 1: Sepsis Protocol ──────────────────────────────────────────────────
print(f"Query: {q1}\n{'─'*60}")
ans1, ctx1, docs1 = rag_response(q1)
print(ans1)
print(f"\n📚 Retrieved {len(docs1)} chunks from Merck Manual (pages: {[d.metadata.get('page', '?') for d in docs1]})")

In [ ]:
# ── Query 2: Appendicitis ─────────────────────────────────────────────────────
print(f"Query: {q2}\n{'─'*60}")
ans2, ctx2, docs2 = rag_response(q2)
print(ans2)
print(f"\n📚 Retrieved {len(docs2)} chunks (pages: {[d.metadata.get('page', '?') for d in docs2]})")

In [ ]:
# ── Query 3: Alopecia Areata ──────────────────────────────────────────────────
print(f"Query: {q3}\n{'─'*60}")
ans3, ctx3, docs3 = rag_response(q3)
print(ans3)
print(f"\n📚 Retrieved {len(docs3)} chunks (pages: {[d.metadata.get('page', '?') for d in docs3]})")

In [ ]:
# ── Query 4: Traumatic Brain Injury ──────────────────────────────────────────
print(f"Query: {q4}\n{'─'*60}")
ans4, ctx4, docs4 = rag_response(q4)
print(ans4)
print(f"\n📚 Retrieved {len(docs4)} chunks (pages: {[d.metadata.get('page', '?') for d in docs4]})")

In [ ]:
# ── Query 5: Leg Fracture ─────────────────────────────────────────────────────
print(f"Query: {q5}\n{'─'*60}")
ans5, ctx5, docs5 = rag_response(q5)
print(ans5)
print(f"\n📚 Retrieved {len(docs5)} chunks (pages: {[d.metadata.get('page', '?') for d in docs5]})")

---
## 7. 📊 Evaluation — LLM-as-a-Judge

We use the **LLM-as-a-Judge** pattern to automatically evaluate RAG response quality on two axes:

| Metric | Definition | Why It Matters |
|---|---|---|
| **Groundedness** | Is the answer supported by the retrieved context? | Catches hallucinations |
| **Relevance** | Does the answer address the user's question? | Ensures clinical utility |

Both are scored **1–5** by the same Mistral model, enabling automated, scalable evaluation.

In [ ]:
# ── Evaluation prompt templates ───────────────────────────────────────────────
GROUNDEDNESS_SYSTEM = """You are a strict medical content evaluator.
Rate how well the AI-generated answer is grounded in the provided medical context.

Scoring rubric (1-5):
    5 — Fully supported; every claim traces back to the context.
    4 — Mostly supported; minor points slightly beyond context.
    3 — Partially supported; some unsupported claims.
    2 — Poorly supported; most claims not in context.
    1 — Not supported at all; contradicts or ignores context.

Respond ONLY with:
Score: <number>
Reason: <one sentence>
"""

RELEVANCE_SYSTEM = """You are a strict medical content evaluator.
Rate how relevant the AI-generated answer is to the user's question.

Scoring rubric (1-5):
    5 — Directly and completely addresses the question.
    4 — Mostly addresses the question with minor gaps.
    3 — Partially addresses; key aspects missing.
    2 — Tangentially related; misses the core question.
    1 — Irrelevant.

Respond ONLY with:
Score: <number>
Reason: <one sentence>
"""

EVAL_USER_TEMPLATE = """Context:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}\n\nEvaluate:"""

print("✅ Evaluation prompts defined.")

In [ ]:
import re

def evaluate(query, answer, context, max_tokens=128, temperature=0):
    """Score a RAG response on groundedness and relevance (1-5 each)."""
    eval_user = EVAL_USER_TEMPLATE.format(
        context=context[:2000], question=query, answer=answer
    )
    
    def _score(system_prompt):
        out = llm(
            prompt=f"{system_prompt}\n\n{eval_user}",
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=0.95, top_k=50, stop=['INST']
        )
        text = out['choices'][0]['text']
        score_match = re.search(r"Score:\s*([1-5])", text, re.IGNORECASE)
        reason_match = re.search(r"Reason:\s*(.+)", text, re.IGNORECASE | re.DOTALL)
        score = int(score_match.group(1)) if score_match else None
        reason = reason_match.group(1).strip()[:200] if reason_match else "N/A"
        return score, reason
    
    g_score, g_reason = _score(GROUNDEDNESS_SYSTEM)
    r_score, r_reason = _score(RELEVANCE_SYSTEM)
    return g_score, g_reason, r_score, r_reason

print("✅ Evaluation function defined.")

In [ ]:
# ── Run evaluation on all 5 queries ──────────────────────────────────────────
eval_data = [
    {"query": q1, "answer": ans1, "context": ctx1},
    {"query": q2, "answer": ans2, "context": ctx2},
    {"query": q3, "answer": ans3, "context": ctx3},
    {"query": q4, "answer": ans4, "context": ctx4},
    {"query": q5, "answer": ans5, "context": ctx5},
]

results = []
for i, item in enumerate(eval_data, 1):
    print(f"\nEvaluating query {i}/5: {item['query'][:60]}...")
    g_score, g_reason, r_score, r_reason = evaluate(
        item["query"], item["answer"], item["context"]
    )
    results.append({
        "Query": f"Q{i}: {item['query'][:50]}...",
        "Groundedness": g_score,
        "Groundedness Reason": g_reason,
        "Relevance": r_score,
        "Relevance Reason": r_reason,
    })
    print(f"  🔗 Groundedness: {g_score}/5 — {g_reason}")
    print(f"  🎯 Relevance:    {r_score}/5 — {r_reason}")

print("\n✅ Evaluation complete.")

In [ ]:
# ── Evaluation summary table ──────────────────────────────────────────────────
eval_df = pd.DataFrame(results)

print("\n📊 EVALUATION SUMMARY TABLE")
print("═" * 80)
display(eval_df[['Query', 'Groundedness', 'Relevance']])

g_scores = eval_df['Groundedness'].dropna()
r_scores = eval_df['Relevance'].dropna()

print(f"\n{'─'*40}")
print(f"Average Groundedness Score : {g_scores.mean():.2f}/5")
print(f"Average Relevance Score    : {r_scores.mean():.2f}/5")
print(f"{'─'*40}")

---
## 8. 💡 Actionable Insights & Business Recommendations

### Key Findings

| Approach | Strengths | Weaknesses |
|---|---|---|
| **Baseline LLM** | Fast, no setup | Hallucinations, generic answers |
| **Prompt Engineering** | Better structure | Still ungrounded |
| **RAG System** | Grounded, specific, citable | Retrieval quality matters |

### Recommendations

1. **Deploy RAG over baseline LLM** for any clinical information retrieval task — the groundedness improvement is substantial and directly reduces patient safety risk.

2. **Tune chunk size and overlap** for your specific use case. Smaller chunks (500 chars) increase precision for drug dosage queries; larger chunks (1500 chars) work better for protocol questions.

3. **Implement hybrid search** (BM25 + semantic) to improve recall for highly technical queries with specific medical terminology.

4. **Add a re-ranking layer** (cross-encoder) to improve precision on the retrieved chunks before generation.

5. **Expand the knowledge base** beyond a single manual — integrating current clinical guidelines (NICE, WHO, CDC) would significantly improve coverage and recency.

6. **Build a feedback loop** — allow clinicians to flag incorrect answers, creating a dataset for continued fine-tuning and evaluation.

### Business Impact

> This system can reduce clinical information retrieval time from **minutes → seconds**, support junior clinicians with senior-level knowledge access, and standardize evidence-based decision-making across departments — directly contributing to improved patient outcomes and reduced medical errors.